# Intro

## Global parameters

In [1]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [2]:
import os, pickle, platform, sys
import numpy as np
import torch

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [9]:
from bowtie import edges2bowtie

### Home made

In [10]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [11]:
from auxiliary_functions import el2ks, bic

In [12]:
from sam_bowtie import block_and_fluxes as bnf

In [13]:
from bowtie_plot_functions import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from bowtie_plot_functions import _fdr as fdr

## Load data

In [14]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'

# Looking into the abyss

In [16]:
guarino_files=[file for file in os.listdir(GUARINO_FOLDER) if not file.startswith('.')]
guarino_files.sort()
guarino_files

['all_dico_labels.txt',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'crisi_dico_labels.pickle',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'ita_elections_dico_labels.pickle',
 'quirinale_dico_0_bowtie_sizes.csv',
 'quirinale_dico_1_bowtie_sizes.csv',
 'quirinale_dico_2_bowtie_sizes.csv',
 'quirinale_dico_3_bowtie_sizes.csv',
 'quirinale_dico_4_bowtie_sizes.csv',
 'quirinale_dico_5_bowtie_sizes.csv',
 'quirinale_dico_6_bowtie_sizes.csv',
 'quirinale_dico_labels.pickle']

In [17]:
bipartite_files=[file for file in os.listdir(BIPARTITE_FOLDER) if not file.startswith('.')]
bipartite_files.sort()
bipartite_files

['crisi_dico_0_bowtie_flows.csv',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_flows.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_flows.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_flows.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_flows.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_0_bowtie_flows.csv',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_flows.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_flows.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_flows.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_flows.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_flows.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_flows.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'quirinale_dico_0_bowtie_flows.csv',
 'quirinale_dico_0_bowtie_sizes.cs

### Name of the various dicos

In [18]:
guarino_files[0]

'all_dico_labels.txt'

In [19]:
with open(GUARINO_FOLDER+guarino_files[0], 'r') as f:
    cacca=f.readlines()


In [20]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [21]:
cacca=parse_dico_names(GUARINO_FOLDER+guarino_files[0])

In [22]:
cacca

{'quirinale': {5: 'journalists & Media',
  2: 'M5S',
  0: 'journalists & IV & Azione & +Europa & Media',
  4: 'Lega & FDI',
  1: 'PD',
  3: 'Media & journalists',
  6: 'FI'},
 'crisi': {1: 'Lega & FDI & FI',
  2: 'M5S & journalists',
  0: 'journalists & IV & Media & Azione & +Europa',
  4: 'Media & journalists',
  3: 'PD'},
 'ita_elections': {1: 'PD & Media & +Europa & journalists',
  2: 'M5S & Media',
  3: 'journalists & IV & Azione',
  0: 'Lega & FDI & FI',
  4: 'journalists & Media (1)',
  5: 'Media',
  6: 'journalists & Media (2)'}}

# Benchmarking: Ita_election, dico3

In [23]:
test_files=os.listdir(TEST_FOLDER)
test_files.sort()

In [24]:
elected_files=[file for file in test_files if file.startswith('ita')]
elected_files

['ita_elections_dico0_qdecm.pkl',
 'ita_elections_dico1_qdecm.pkl',
 'ita_elections_dico2_decm.pkl',
 'ita_elections_dico2_decm_final.pkl',
 'ita_elections_dico2_qdecm.pkl',
 'ita_elections_dico3_decm.pkl',
 'ita_elections_dico3_decm_final.pkl',
 'ita_elections_dico3_decm_final_0.pkl',
 'ita_elections_dico3_dwcm_final.pkl',
 'ita_elections_dico3_qdecm.pkl',
 'ita_elections_dico4_qdecm.pkl',
 'ita_elections_dico5_qdecm.pkl',
 'ita_elections_dico6_qdecm.pkl']

In [25]:
for file in elected_files:
    if 'dico3' in file:
        with open(TEST_FOLDER+file, 'rb') as f:
            cacca=pickle.load(f)
        file_name_friendly=file[14:25].strip('.')
        if cacca.sol.mre<10**-5:
            print(f"{file_name_friendly}: Ok, MRE={cacca.sol.mre:.1e}")
        else:
            print(f"{file_name_friendly}: Bad! MRE={cacca.sol.mre:.1e} ")

    

dico3_decm: Bad! MRE=2.7e-03 
dico3_decm_: Bad! MRE=2.7e-03 
dico3_decm_: Bad! MRE=1.9e-03 
dico3_dwcm_: Ok, MRE=1.1e-06
dico3_qdecm: Ok, MRE=4.7e-06


Ok, so far no decm converge with MRE<10^-5. Not good... Nevertheless, let us proceed with the calculation and then substitute the proper values once we find convergence.

## Comparison

In [37]:
ita_el_dict=defaultdict(dict)

for file in elected_files:
    dico=int(file.split('dico')[1][0])
    model=file.split('cm')[0].split('_')[-1]+'cm'
    try:
        with open(TEST_FOLDER+file, 'rb') as f:
            cacca=pickle.load(f)
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue
    
    if cacca.sol.converged:
        ita_el_dict[dico][model]=cacca
        

In [38]:
ita_el_dict

defaultdict(dict,
            {0: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x176872860>},
             1: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x176adcbe0>},
             2: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x1769370a0>},
             3: {'dwcm': <dcms.models.dwcm.DWCMModel at 0x176871d80>,
              'qdecm': <dcms.models.qdecm.qDECMModel at 0x1764c2ef0>},
             4: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x176870bb0>},
             5: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x1769401c0>},
             6: {'qdecm': <dcms.models.qdecm.qDECMModel at 0x176940850>}})

### Dico3

In [41]:
ita_el_dict[3].keys()

dict_keys(['dwcm', 'qdecm'])

In [40]:
dwcm_theta=ita_el_dict[3]['dwcm'].sol.best_theta
qdecm_theta=ita_el_dict[3]['qdecm'].sol.best_theta

In [45]:
N=len(dwcm_theta)//2
2*len(dwcm_theta)==len(qdecm_theta)

True

### Likelihood

In [48]:
dwcm_ll=ita_el_dict[3]['dwcm'].neg_log_likelihood(dwcm_theta)

In [49]:
qdecm_ll=ita_el_dict[3]['qdecm'].neg_log_likelihood_strength(qdecm_theta[:2*N], qdecm_theta[2*N:])

In [50]:
f'{dwcm_ll:.2e}, {qdecm_ll:.2e}'

'1.43e+06, 2.55e+05'

### BIC

In [ ]:
dwcm_bic=2 * dwcm_ll + 2*N * np.log(N*(N-1)/2)
qdecm_bic=2 * qdecm_ll + 4*N * np.log(N*(N-1)/2)
f'{dwcm_bic:.2e}, {qdecm_bic:.2e}'

'3.97e+06, 2.74e+06'

The minimum one wins!

### Calculation time

In [55]:
ita_el_dict[3]['dwcm'].sol.elapsed_time

1.9882352500007983

In [54]:
ita_el_dict[3]['qdecm'].sol.elapsed_time

278.7972812910011

## Plots

### Lagrangian multipliers comparison

In [ ]:
fig, axs= plt.subplots(2, 2, figsize=(12, 12), constrained_layout=True)
colors=['navy', 'darkorange', 'darkred', 'magenta']
labels_decm=[r'$\theta^{OUT}_{DECM}$', r'$\theta^{IN}_{DECM}$', r'$\eta^{OUT}_{DECM}$', r'$\eta^{IN}_{DECM}$']
labels_qdecm=[r'$\theta^{OUT}_{qDECM}$', r'$\theta^{IN}_{qDECM}$', r'$\eta^{OUT}_{qDECM}$', r'$\eta^{IN}_{qDECM}$']
for i in range(2):
    for j in range(2):
        _decm_theta=decm_theta[i*l_theta//4+j*l_theta//2:(i+1)*l_theta//4+j*l_theta//2]
        _qdecm_theta=qdecm_theta[i*l_theta//4+j*l_theta//2:(i+1)*l_theta//4+j*l_theta//2]
        axs[i, j].scatter(_decm_theta, _qdecm_theta, color=colors[i + 2*j])
        axs[i, j].plot([-20,100], [-20,100], '--', color='gray')
        axs[i, j].set_xlim([-20, 100])
        axs[i, j].set_ylim([-20, 100])
        axs[i, j].set_xlabel(labels_decm[i + 2*j], fontsize=16)
        axs[i, j].set_ylabel(labels_qdecm[i + 2*j], fontsize=16)
plt.show()

### Probabilities and expected weights comparison

In [ ]:
# colors and labels
colors=['orchid','darkcyan']
labels_decm=[r'$p_{DECM}$', r'$\langle w\rangle_{DECM}$']
labels_qdecm=[r'$p_{qDECM}$', r'$\langle w\rangle_{qDECM}$']

In [ ]:
# probability matrices
p_decm=crisi_dico2_sols['decm'].pij_matrix(crisi_dico2_sols['decm'].sol.theta).flatten()
p_qdecm=crisi_dico2_sols['qdecm'].pij_matrix(crisi_dico2_sols['qdecm'].sol.theta[:l_theta//2]).flatten()

In [ ]:
# weight matrices
w_decm=crisi_dico2_sols['decm'].wij_matrix(crisi_dico2_sols['decm'].sol.theta).flatten()
w_qdecm=crisi_dico2_sols['qdecm'].wij_matrix_conditioned(crisi_dico2_sols['qdecm'].sol.theta[:l_theta//2], crisi_dico2_sols['qdecm'].sol.theta[l_theta//2:]).flatten()


In [ ]:
len(p_decm), len(p_qdecm), len(w_decm), len(w_qdecm)
print(f"Length of p_decm: {len(p_decm):.2e}")

In [ ]:
f'{(l_theta//4)**2:.2e}'

In [ ]:
sample_size=10**6

In [ ]:
selected_points=np.random.choice(np.arange(len(p_decm)), size=sample_size, replace=False)

In [ ]:
fig, axs= plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

axs[0].scatter(p_decm[selected_points], p_qdecm[selected_points], color=colors[0])
axs[0].plot([-1,2], [-1,2], '--', color='gray')
axs[0].set_xlim([-0.05, 1.05])
axs[0].set_ylim([-0.05, 1.05])

axs[1].scatter(w_decm[selected_points], w_qdecm[selected_points], color=colors[1])
axs[1].plot([-1,10], [-1,10], '--', color='gray')
axs[1].set_xlim([-0.1, 4])
axs[1].set_ylim([-0.1, 4])

    
for i in range(2):
    axs[i].set_xlabel(labels_decm[i], fontsize=16)
    axs[i].set_ylabel(labels_qdecm[i], fontsize=16)
plt.show()

## Maximum "error"

In [ ]:
w_mask=w_decm>0
p_mask=p_decm>0

In [ ]:
torch.all(p_mask==w_mask)

In [ ]:
w_mask.sum()

In [ ]:
torch.all(p_qdecm[p_decm==0]==0)

In [ ]:
torch.all(w_qdecm[w_decm==0]==0)

So, all the zero entries of p(DECM) are also zero according to qDECM. Good...

### Maximum absolute and relative error on $p$

In [ ]:
abs_error_p=torch.abs(p_decm[w_mask]-p_qdecm[w_mask])
torch.max(abs_error_p)

In [ ]:
where_max_abs_err_p=torch.where(abs_error_p==torch.max(abs_error_p))[0][0]

In [ ]:
p_decm[w_mask][where_max_abs_err_p], p_qdecm[w_mask][where_max_abs_err_p]

In [ ]:
torch.max(abs_error_p)/p_decm[w_mask][where_max_abs_err_p]

In [ ]:
rel_error_p=torch.abs(p_decm[w_mask]-p_qdecm[w_mask])/p_decm[w_mask]
torch.max(rel_error_p)

In [ ]:
where_max_rel_err_p=torch.where(rel_error_p==torch.max(rel_error_p))[0][0]

In [ ]:
p_decm[w_mask][where_max_rel_err_p], p_qdecm[w_mask][where_max_rel_err_p]

In [ ]:
abs_error_p[where_max_rel_err_p]

### Maximum absolute and relative error on $w$

In [ ]:
abs_error_w=torch.abs(w_decm[w_mask]-w_qdecm[w_mask])
torch.max(abs_error_w)

In [ ]:
where_max_abs_err_w=torch.where(abs_error_w==torch.max(abs_error_w))[0][0]

In [ ]:
w_decm[w_mask][where_max_abs_err_w], w_qdecm[w_mask][where_max_abs_err_w]

In [ ]:
torch.max(abs_error_w)/w_decm[w_mask][where_max_abs_err_w]

In [ ]:
rel_error_w=torch.abs(w_decm[w_mask]-w_qdecm[w_mask])/w_decm[w_mask]
torch.max(rel_error_w)

In [ ]:
where_max_rel_err_w=torch.where(rel_error_w==torch.max(rel_error_w))[0][0]

In [ ]:
w_decm[w_mask][where_max_rel_err_w], w_qdecm[w_mask][where_max_rel_err_w]

In [ ]:
abs_error_w[where_max_rel_err_w]

In [ ]:
where_max_rel_err_w, where_max_rel_err_p

This is quite suspect...

# Summarising
1. so far, all algorithms follows a local Newton method, i.e. a Newton in which, instead of calculating the inverse of the Hessian, all entries are considered as independent (they are not). It works on every models for all datasets, but for DECM; In the cas eof DECM, it properly works on all test datasets (i.e. N~10k).
2. the only empirical network on which the DECM converge is crisis dico2. It converged by accident: the evolution of the MRE jumps around savagely until it find the proper path and then converges immediately. The same procedure was applied with no success on other dataset. All other tricks (backtracking, 2x2, different update phases) implemented did not work;
3. on crisi dico2 differences are impressive: the relative error on both the expected weights and the expected topology (i.e. $p$) is up to 5 (I mean, 500%), which is quite high. **A proper comparison should show how the p-value calculated with DECM and qDECM differ for our case, i.e. for the validation of the fluxes and the dimension of the blocks in the bowtie**; 
4. as DECM seems to goverge _extremely slowly_ to the proper solution, I can decide that I can wait for a couple of datasets considered as benchmarks, i.e. IV and DX (as they have the most marked bowtie structure) for the dataset used as benchmark, i.e. ita_elections. 
5. the likelihood, as calculated so far, needs to be checked. Consider also that on the networks that we decide to use as benchmarks, a model selection should be run **and** a fine justification for using qDECM instead of DECM should be found, even if BIC could never win (unless we used the _ad hoc_ defined generalized likelihood).
6. Hooray! o_o

# Summarising, 2nd season
1. it should be added to the draft. Or not?
2. should the fact the none of the other methods tested converged should be mentioned? Furthermore, it seems that on the other networks intended to be used as benchmarks, the convergence fails (related also to point 4.).
3. **DO IT!**
4. **FUCK!** None of the converge, not even slowly: after a clear decrease, the system started jumping wildly, moslty in the wrong direction. 
5. It seems correct. (*_I should check it again, as I do not remember_*) In the case of the qDECM, the generalized likelihood is calculated.
6. Even worse...